In [1]:
import torch
import torch.nn as nn
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
import warnings
from tqdm import tqdm
import os
import json
from torch_geometric.data import Data, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence


# Ignore specific warnings
warnings.filterwarnings("ignore")

class FUSION(torch.nn.Module):
    def __init__(self, hidden_channels, input_size):
        super(FUSION, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GATConv(input_size, hidden_channels)
        self.conv2 = GATConv(hidden_channels, hidden_channels)
        self.conv3 = GATConv(hidden_channels, hidden_channels)
        # self.conv4 = GCNConv(input_size, 64)
        self.lin = Linear(hidden_channels, 3)

    def forward(self, x, edge_index, batch):
        # 1. Obtain node embeddings
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)
        # x = x.relu()
        # x = self.conv4(x, edge_index)

        # 2. Readout layer
        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        # 3. Apply a final classifier
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return x

In [2]:

def load_json_folder(folder_path):
    data_array = []

    # Iterate through files in the folder
    for filename in tqdm(os.listdir(folder_path)):
        file_path = os.path.join(folder_path, filename)

        # Check if the file is a JSON file
        if filename.endswith(".json"):
            # Read and load the JSON file
            with open(file_path, 'r') as json_file:
                data = json.load(json_file)

            # Append the loaded data to the array
            data_array.append(data)


    return data_array
def convert_to_one_hot(label_tensor, num_classes):
    """
    Convert a label tensor to its one-hot encoded representation.

    Args:
    - label_tensor (torch.Tensor): Tensor containing the labels.
    - num_classes (int): Number of classes.

    Returns:
    - torch.Tensor: One-hot encoded tensor.
    """
    # Initialize the one-hot encoded tensor
    one_hot_tensor = torch.zeros(len(label_tensor), num_classes)

    # Fill in the one-hot encoded tensor
    one_hot_tensor[range(len(label_tensor)), label_tensor] = 1

    return one_hot_tensor
def train(model, optimizer, criterion, loader, device):
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model.train()
    total_loss = 0
    y_true = []
    y_pred = []

    for data in loader:  # Iterate in batches over the train dataset.
        # Ensure data is on the correct device and type
        data.x = data.x.to(device, dtype=torch.float32)  # Convert to float32 and ensure on GPU
        data.edge_index = data.edge_index.to(device, dtype=torch.long)  # Ensure edge_index is long and on GPU
        data.y = data.y.to(device, dtype=torch.long)  # Ensure y is long and on GPU
        data.batch = data.batch.to(device) if data.batch is not None else None  # Handle batch index for batched graphs
        # batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
        
        # Forward pass
        out = model(data.x, data.edge_index, data.batch)  # Perform a single forward pass
        loss = criterion(out, data.y)  # Compute loss (CrossEntropyLoss expects long labels, not one-hot)
        loss.backward()  # Derive gradients
        optimizer.step()  # Update parameters based on gradients
        optimizer.zero_grad()  # Clear gradients

        # Compute predictions for metrics
        pred = out.argmax(dim=1)  # Get predicted class
        y_true.extend(data.y.cpu().tolist())  # Move to CPU for sklearn metrics
        y_pred.extend(pred.cpu().tolist())  # Move to CPU for sklearn metrics
        total_loss += loss.item()  # Accumulate loss (scale by batch size)

    # Calculate metrics
    average_loss = total_loss / len(loader)  # Average loss per graph
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    return average_loss, accuracy, precision, recall, f1, model
    
def test(model, loader):
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    model.eval()
    y_true = []
    y_pred = []
    correct = 0

    for data in loader:  # Iterate in batches over the training/test dataset.
        data.edge_index = data.edge_index.to(torch.long)
        data.x = data.x.to(model.parameters().__next__().dtype)
        # batch = torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        out = model(data.x, data.edge_index, data.batch)
        pred = out.argmax(dim=1)  # Use the class with the highest probability.
        y_true.extend(data.y.tolist())
        y_pred.extend(pred.tolist())
        # correct += int((pred == data.y).sum())  # Check against ground-truth labels.

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)  # or 'micro' or 'weighted'
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    # Compute accuracy
    # accuracy = correct / len(loader.dataset)

    return accuracy, precision, recall, f1

In [3]:

# Assuming train, test, load_json_folder, and FUSION are defined elsewhere
# from your_module import train, test, load_json_folder, FUSION

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load training data
train_data = load_json_folder("data/delegatecall_train")
Train_data = []
for data in train_data:
    x = torch.tensor(data["node_feature"], dtype=torch.float64).to(device)
    inputsize = x.shape[1]
    sources = [int(num) for num in data["edge_index"][0]]
    target = [int(num) for num in data["edge_index"][1]]
    edges = torch.tensor([sources, target]).to(device)
    y = torch.tensor([data["label"]], dtype=torch.long).to(device)
    Train_data.append(Data(x=x, edge_index=edges, y=y))

# Load test data
Test_data = []
test_data = load_json_folder("data/delegatecall_test")
for data in test_data:
    x = torch.tensor(data["node_feature"], dtype=torch.float64).to(device)
    sources = [int(num) for num in data["edge_index"][0]]
    target = [int(num) for num in data["edge_index"][1]]
    edges = torch.tensor([sources, target]).to(device)
    y = torch.tensor([data["label"]], dtype=torch.long).to(device)
    Test_data.append(Data(x=x, edge_index=edges, y=y))

# Create DataLoader instances
train_loader = DataLoader(Train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(Test_data, batch_size=32, shuffle=False)

# Initialize model, criterion, and optimizer
model = FUSION(hidden_channels=32, input_size=inputsize).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Early stopping parameters
patience = 5  # Number of epochs to wait for improvement
best_accuracy = 0.0
epochs_no_improve = 0
best_model_path = 'model/delegatecall_model_best.pth'

# Training and testing loop
for epoch in range(200):
    # Training
    model.train()
    train_loss, train_accuracy, train_precision, train_recall, train_f1, model = train(model, optimizer, criterion, train_loader, device)

    # Testing
    model.eval()
    test_accuracy, test_precision, test_recall, test_f1 = test(model, test_loader)

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1} | Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | "
              f"Train Precision: {train_precision:.4f} | Train Recall: {train_recall:.4f} | Train F1: {train_f1:.4f}")
        print(f"Test Accuracy: {test_accuracy:.4f} | Test Precision: {test_precision:.4f} | "
              f"Test Recall: {test_recall:.4f} | Test F1: {test_f1:.4f}")

    # # Early stopping logic
    # if test_accuracy > best_accuracy:
    #     best_accuracy = test_accuracy
    #     epochs_no_improve = 0
    #     # Save the best model
    #     torch.save(model.state_dict(), best_model_path)
    #     # print(f"New best test accuracy: {best_accuracy:.4f}, model saved to {best_model_path}")
    # else:
    #     epochs_no_improve += 1
    #     # print(f"No improvement in test accuracy. Epochs without improvement: {epochs_no_improve}")

    # # Check for early stopping
    # if epochs_no_improve >= patience:
    #     # print(f"Early stopping triggered after {epoch + 1} epochs. No improvement in test accuracy for {patience} epochs.")
    #     break

# Load and evaluate the best model
# model.load_state_dict(torch.load(best_model_path, map_location=device))
# model.eval()
# final_accuracy, final_precision, final_recall, final_f1 = test(model, test_loader)
# print(f"Final Best Model | Accuracy: {final_accuracy:.4f} | Precision: {final_precision:.4f} | "
#       f"Recall: {final_recall:.4f} | F1 Score: {final_f1:.4f}")
# print(f"Best model saved at: {best_model_path}")

Using device: cuda


100%|██████████| 36/36 [00:00<00:00, 2329.67it/s]


Epoch 10 | Train Loss: 0.6638 | Train Accuracy: 0.6522 | Train Precision: 0.5731 | Train Recall: 0.5573 | Train F1: 0.5569
Test Accuracy: 0.7500 | Test Precision: 0.8000 | Test Recall: 0.3333 | Test F1: 0.4706
Epoch 20 | Train Loss: 0.5758 | Train Accuracy: 0.7899 | Train Precision: 0.7590 | Train Recall: 0.7490 | Train F1: 0.7535
Test Accuracy: 0.8056 | Test Precision: 0.6923 | Test Recall: 0.7500 | Test F1: 0.7200
Epoch 30 | Train Loss: 0.4375 | Train Accuracy: 0.8116 | Train Precision: 0.7917 | Train Recall: 0.7589 | Train F1: 0.7712
Test Accuracy: 0.8056 | Test Precision: 0.7273 | Test Recall: 0.6667 | Test F1: 0.6957
Epoch 40 | Train Loss: 0.3579 | Train Accuracy: 0.8696 | Train Precision: 0.8668 | Train Recall: 0.8257 | Train F1: 0.8416
Test Accuracy: 0.7778 | Test Precision: 1.0000 | Test Recall: 0.3333 | Test F1: 0.5000
Epoch 50 | Train Loss: 0.2878 | Train Accuracy: 0.8986 | Train Precision: 0.8832 | Train Recall: 0.8832 | Train F1: 0.8832
Test Accuracy: 0.9167 | Test Precisio

In [4]:
for epoch in range(200):
    # Training
    model.train()
    train_loss, train_accuracy, train_precision, train_recall, train_f1, model = train(model, optimizer, criterion, train_loader, device)

    # Testing
    model.eval()
    test_accuracy, test_precision, test_recall, test_f1 = test(model, test_loader)

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1} | Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | "
              f"Train Precision: {train_precision:.4f} | Train Recall: {train_recall:.4f} | Train F1: {train_f1:.4f}")
        print(f"Test Accuracy: {test_accuracy:.4f} | Test Precision: {test_precision:.4f} | "
              f"Test Recall: {test_recall:.4f} | Test F1: {test_f1:.4f}")

Epoch 10 | Train Loss: 0.2884 | Train Accuracy: 0.8696 | Train Precision: 0.8564 | Train Recall: 0.8378 | Train F1: 0.8460
Test Accuracy: 0.8889 | Test Precision: 0.9000 | Test Recall: 0.7500 | Test F1: 0.8182
Epoch 20 | Train Loss: 0.1574 | Train Accuracy: 0.9493 | Train Precision: 0.9394 | Train Recall: 0.9446 | Train F1: 0.9420
Test Accuracy: 0.8889 | Test Precision: 1.0000 | Test Recall: 0.6667 | Test F1: 0.8000
Epoch 30 | Train Loss: 0.1463 | Train Accuracy: 0.9420 | Train Precision: 0.9608 | Train Recall: 0.9091 | Train F1: 0.9296
Test Accuracy: 0.9167 | Test Precision: 1.0000 | Test Recall: 0.7500 | Test F1: 0.8571
Epoch 40 | Train Loss: 0.0820 | Train Accuracy: 0.9710 | Train Precision: 0.9583 | Train Recall: 0.9787 | Train F1: 0.9674
Test Accuracy: 0.9167 | Test Precision: 1.0000 | Test Recall: 0.7500 | Test F1: 0.8571
Epoch 50 | Train Loss: 0.0397 | Train Accuracy: 0.9928 | Train Precision: 0.9889 | Train Recall: 0.9947 | Train F1: 0.9917
Test Accuracy: 0.8889 | Test Precisio

In [3]:
#train

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
train_data = load_json_folder("data/delegatecall_train")
Train_data = []
for data in train_data:
    x = torch.tensor(data["node_feature"], dtype=torch.float64).to(device)  # Move to GPU
    inputsize = x.shape[1]
    sources = [int(num) for num in data["edge_index"][0]]
    target = [int(num) for num in data["edge_index"][1]]
    edges = torch.tensor([sources, target]).to(device)  # Move to GPU
    y = torch.tensor([data["label"]], dtype=torch.long).to(device)  # Move to GPU
    Train_data.append(Data(x=x, edge_index=edges, y=y))

# Create DataLoader instances
train_loader = DataLoader(Train_data, batch_size=32, shuffle=True)

# Initialize model, criterion, and optimizer
model = FUSION(hidden_channels=32, input_size=inputsize).to(device)  # Move model to GPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# Training loop
for epoch in range(30):
    # Training
    train_loss, a, p, r, f, model = train(model, optimizer, criterion, train_loader,device)

    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1} Train Loss: {train_loss:.4f} Train Accuracy: {a:.4f}, Precision: {p:.4f}, Recall: {r:.4f}, F1 Score: {f:.4f}")
        # torch.save(model.state_dict(), 'model/'+args.model_name+'.pth')
        # print("save model OK")

torch.save(model.state_dict(), 'model/'+"delegatecall_model"+'.pth')
print("save model OK")

Using device: cuda


100%|██████████| 138/138 [00:00<00:00, 1092.39it/s]


Epoch 10 Train Loss: 0.3910 Train Accuracy: 0.8478, Precision: 0.8415, Recall: 0.7976, F1 Score: 0.8138
Epoch 20 Train Loss: 0.2179 Train Accuracy: 0.8986, Precision: 0.9044, Recall: 0.8590, F1 Score: 0.8768
Epoch 30 Train Loss: 0.1374 Train Accuracy: 0.9493, Precision: 0.9569, Recall: 0.9265, F1 Score: 0.9397
save model OK


In [7]:
#test

# Check if model_path and test_data_folder are provided
# if args.model_path is None or args.test_data_folder is None:
#     raise ValueError("In test mode, --model_path and --test_data_folder must be provided.")

# Load test data

Test_data = []
test_data = load_json_folder("data/delegatecall_test")
for data in test_data:
    x = torch.tensor(data["node_feature"], dtype=torch.float64).to(device)  # Move to GPU
    sources = [int(num) for num in data["edge_index"][0]]
    target = [int(num) for num in data["edge_index"][1]]
    edges = torch.tensor([sources, target]).to(device)  # Move to GPU
    y = torch.tensor([data["label"]], dtype=torch.long).to(device)  # Move to GPU

    Test_data.append(Data(x=x, edge_index=edges, y=y))

# Create DataLoader instance for test data
test_loader = DataLoader(Test_data, batch_size=32, shuffle=False)

# Initialize model and criterion
model = FUSION(hidden_channels=32, input_size=77).to(device)  # Move model to GPU
model.load_state_dict(torch.load("model/delegatecall_model.pth", map_location=device))  # Load model to GPU
model.eval()

# Testing
a, p, r, f = test(model, test_loader)
print(f"Accuracy: {a:.4f}, Precision: {p:.4f}, Recall: {r:.4f}, F1 Score: {f:.4f}")

100%|██████████| 36/36 [00:00<00:00, 1100.95it/s]

Accuracy: 0.8611, Precision: 0.8182, Recall: 0.7500, F1 Score: 0.7826
